# 最大連結成分に含まれる H₂ の占有率依存

H₂は、結合をちょうど2本共有する分子ペアの数です。各試料で分子数が最大の連結成分を1個選び、その成分内の H₂ を `H2_largest` として集計します。同率最大では成分IDが最小のものを選びます。

全体のH₂のうち最大成分に入った割合（`H2_largest / H2`）と、全分子のうち最大成分に入った割合（`N1 / N`）を比較できます。成分内の全末端に占める割合は `H2_largest / N1` です。3つの割合はそれぞれ100倍して％で表示します。

初期設定は保存済みの実試料1件です。試せたら `MODE` を `"pilot"`（各条件1試料）または `"full"`（各条件100試料）に変えて設定セルから実行します。対象はL=32・64・128、目標占有率0.10〜0.70の0.01刻みです。pilotとfullは同じ出力先を使い、保存済みの行を再利用します。分割バックアップは必要な試料に到達するまで順番に読みます。

試料別CSV、条件平均・標準誤差のCSV、4パネルの比較図を出力します。ゼロ分母の比率は空欄、1試料での標準誤差も空欄です。


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

MODE = "smoke"  # smoke / pilot / full
DRIVE_ROOT = Path("/content/drive/MyDrive/tetra-peg-network")
STUDY = DRIVE_ROOT / "backup" / "h2_relation_fine_compact"
ARCHIVE = DRIVE_ROOT / "backup" / "h2_archive_20261028"
SAMPLES = STUDY / "results_fe428bde2d" / "samples.csv"
OUTPUT_ROOT = DRIVE_ROOT / "cycle46_report" / "h2_largest"


In [ ]:
import csv
from pathlib import Path
import subprocess
import sys
import tempfile
from IPython.display import Image, display

if MODE not in {"smoke", "pilot", "full"}:
    raise ValueError("MODE must be smoke, pilot, or full")
case = "L128_phi0.10_i000000"
candidates = [SAMPLES, SAMPLES.with_name("upload_h2_validation.zip")]
samples = next((p for p in candidates if p.is_file()), None)
if samples is None:
    raise FileNotFoundError(f"試料表の場所を SAMPLES に設定してください。確認先 {SAMPLES}")

loose = STUDY / "cases" / case / "clusters.csv.gz"
if MODE == "smoke" and loose.is_file():
    source = ["--study", str(STUDY)]
elif (ARCHIVE / "h2.tar.part-000").is_file():
    source = ["--archive", str(ARCHIVE)]
    print("分割バックアップを順番に読みます。", flush=True)
else:
    source = ["--study", str(STUDY)]

output = OUTPUT_ROOT / ("smoke" if MODE == "smoke" else "comparison")
with tempfile.TemporaryDirectory(prefix="h2_largest_") as temp:
    repo = Path(temp) / "tetra-peg-network"
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch",
                    "codex/ab-saw-cycle46-colab",
                    "https://github.com/takakei510/tetra-peg-network.git", str(repo)], check=True)
    commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    print(f"GitHub commit: {commit}", flush=True)
    cmd = [sys.executable, str(repo / "scripts/analysis/analyze_h2_largest.py"),
           "--samples", str(samples), "--output", str(output), *source]
    if MODE != "full":
        cmd += ["--replicates", "1"]
    if MODE == "smoke":
        cmd += ["--case", case]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, cmd)
        finally:
            if process.poll() is None:
                process.terminate()

if MODE == "smoke":
    with (output / "h2_largest_samples.csv").open(encoding="utf-8-sig") as handle:
        row = next(csv.DictReader(handle))
    actual = (int(row["N1"]), int(row["H2_largest"]), int(row["H2"]))
    if row["case_id"] != case or actual != (12, 1, 195):
        raise ValueError(f"実試料の参照値に一致しません {actual}")
    print("実試料1件を確認。最大成分12分子、最大成分内H₂は1組、全体は195組。")
else:
    display(Image(filename=str(output / "h2_largest_vs_occupancy.png")))
print(f"出力先 {output}")
